# Sentiment Analysis Pipeline — FinBERT
**Stocks:** RELIANCE, HDFCBANK, INFY, BHARTIARTL, HUL, MM  
**Pipeline:** Filter English → FinBERT Scoring → Monthly Aggregation → Save

In [1]:
# Install required packages if not already installed
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers', 'torch', '-q'])


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


CompletedProcess(args=['/usr/local/bin/python3', '-m', 'pip', 'install', 'transformers', 'torch', '-q'], returncode=0)

In [2]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax
from tqdm.notebook import tqdm

# ── MPS fallback: allows unsupported ops to fall back to CPU silently ─
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# ── Config ────────────────────────────────────────────────────────────
RAW_DIR = 'data_1/raw'
OUT_DIR = 'data_1/sentiment'
os.makedirs(OUT_DIR, exist_ok=True)

STOCKS  = ['RELIANCE', 'HDFCBANK', 'INFY', 'BHARTIARTL', 'HUL', 'MM']
BATCH   = 32    # articles per batch (reduce to 16 if memory pressure)
MAX_LEN = 512   # FinBERT max token length

# ── Device: MPS (Apple Silicon GPU) > CUDA > CPU ──────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print('✅ Using Apple Silicon GPU (MPS)')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print('✅ Using CUDA GPU')
else:
    DEVICE = torch.device('cpu')
    print('⚠️  No GPU found — using CPU (will be slow)')

print(f'PyTorch version : {torch.__version__}')
print(f'Device selected : {DEVICE}')

✅ Using Apple Silicon GPU (MPS)
PyTorch version : 2.10.0
Device selected : mps


## Step 1 — Load Raw Data & Filter English Only

In [3]:
def load_and_filter_english(ticker):
    path = os.path.join(RAW_DIR, f'NEWS_{ticker}_RAW.csv')
    df = pd.read_csv(path)
    total = len(df)

    # Parse date
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date'])

    # Filter to English only
    df_en = df[df['language'].str.lower() == 'english'].copy()

    # Drop rows with empty titles
    df_en = df_en[df_en['title'].notna() & (df_en['title'].str.strip() != '')]

    # Keep only Jan 2020 – Dec 2025
    df_en = df_en[(df_en['date'] >= '2020-01-01') & (df_en['date'] <= '2025-12-31')]
    df_en = df_en.sort_values('date').reset_index(drop=True)

    print(f'{ticker}: {total:,} total → {len(df_en):,} English articles '
          f'({total - len(df_en):,} non-English removed)')
    return df_en

# Preview all stocks
print('=' * 60)
for s in STOCKS:
    load_and_filter_english(s)
print('=' * 60)

RELIANCE: 13,585 total → 10,937 English articles (2,648 non-English removed)
HDFCBANK: 13,139 total → 10,864 English articles (2,275 non-English removed)
INFY: 14,691 total → 12,214 English articles (2,477 non-English removed)
BHARTIARTL: 12,435 total → 10,423 English articles (2,012 non-English removed)
HUL: 11,352 total → 10,958 English articles (394 non-English removed)
MM: 13,024 total → 12,240 English articles (784 non-English removed)


## Step 2 — Load FinBERT Model

In [4]:
MODEL_NAME = 'ProsusAI/finbert'
print(f'Loading {MODEL_NAME} ...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model     = model.to(DEVICE)
model.eval()

# Label order from FinBERT: positive=0, negative=1, neutral=2
LABELS = model.config.id2label
print(f'Label mapping: {LABELS}')
print('Model ready.')

Loading ProsusAI/finbert ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Label mapping: {0: 'positive', 1: 'negative', 2: 'neutral'}
Model ready.


## Step 3 — FinBERT Scoring Function

In [5]:
def score_batch(titles):
    encoded = tokenizer(
        titles,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors='pt'
    )
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

    with torch.no_grad():
        logits = model(**encoded).logits
    probs = softmax(logits, dim=1).cpu().numpy()

    # Build once
    label_map = {str(v).lower(): i for i, v in LABELS.items()}

    results = []
    for row in probs:
        pos = float(row[label_map.get('positive', 0)])
        neg = float(row[label_map.get('negative', 1)])
        neu = float(row[label_map.get('neutral',  2)])
        results.append((pos, neg, neu))
    return results


def run_finbert(df):
    """Score all titles in df, return df with sentiment columns added."""
    titles = df['title'].tolist()
    all_scores = []

    for i in tqdm(range(0, len(titles), BATCH), desc='FinBERT batches'):
        batch = titles[i : i + BATCH]
        all_scores.extend(score_batch(batch))

    df = df.copy()
    df['sentiment_positive'] = [s[0] for s in all_scores]
    df['sentiment_negative'] = [s[1] for s in all_scores]
    df['sentiment_neutral']  = [s[2] for s in all_scores]
    # Net score: positive - negative, range [-1, +1]
    df['sentiment_score']    = df['sentiment_positive'] - df['sentiment_negative']
    return df

## Step 4 — Daily Aggregation Function

In [6]:
def aggregate_daily(df_scored):
    """Aggregate article-level scores to DAILY sentiment (calendar day)."""
    df_scored = df_scored.copy()

    # Ensure datetime
    df_scored["date"] = pd.to_datetime(df_scored["date"], errors="coerce")
    df_scored = df_scored.dropna(subset=["date"])

    # Remove time part (keep only date)
    df_scored["date"] = df_scored["date"].dt.normalize()

    daily = df_scored.groupby("date").agg(
        article_count      = ("sentiment_score",    "count"),
        sentiment_score    = ("sentiment_score",    "mean"),  # mean(pos-neg)
        sentiment_positive = ("sentiment_positive", "mean"),
        sentiment_negative = ("sentiment_negative", "mean"),
        sentiment_neutral  = ("sentiment_neutral",  "mean"),
    ).reset_index()

    daily = daily.sort_values("date").reset_index(drop=True)
    return daily

## Step 5 — Run Full Pipeline for All Stocks

In [7]:
summary = []

for ticker in STOCKS:
    print(f'\n{"="*55}')
    print(f'Processing {ticker} ...')

    # 1. Load & filter English
    df_en = load_and_filter_english(ticker)

    # 2. Run FinBERT
    df_scored = run_finbert(df_en)

    # 3. Save article-level scores
    art_path = os.path.join(OUT_DIR, f'SENTIMENT_{ticker}_ARTICLES.csv')
    df_scored.to_csv(art_path, index=False)
    print(f'  Article-level saved → {art_path}')

    # 4. Aggregate daily
    daily = aggregate_daily(df_scored)

    # 5. Save daily scores
    day_path = os.path.join(OUT_DIR, f'SENTIMENT_{ticker}_DAILY.csv')
    daily.to_csv(day_path, index=False)
    print(f'  Daily scores saved → {day_path}')
    print(f'  Days covered: {len(daily)} | '
        f'Avg sentiment: {daily["sentiment_score"].mean():.4f}')

    summary.append({
        'ticker':         ticker,
        'articles_used':  len(df_en),
        'days_covered':   len(daily),
        'avg_sentiment':  round(daily['sentiment_score'].mean(), 4),
    })

print(f'\n{"="*55}')
print('ALL STOCKS COMPLETE')


Processing RELIANCE ...
RELIANCE: 13,585 total → 10,937 English articles (2,648 non-English removed)


FinBERT batches:   0%|          | 0/342 [00:00<?, ?it/s]

  Article-level saved → data_1/sentiment/SENTIMENT_RELIANCE_ARTICLES.csv
  Daily scores saved → data_1/sentiment/SENTIMENT_RELIANCE_DAILY.csv
  Days covered: 409 | Avg sentiment: 0.0965

Processing HDFCBANK ...
HDFCBANK: 13,139 total → 10,864 English articles (2,275 non-English removed)


FinBERT batches:   0%|          | 0/340 [00:00<?, ?it/s]

  Article-level saved → data_1/sentiment/SENTIMENT_HDFCBANK_ARTICLES.csv
  Daily scores saved → data_1/sentiment/SENTIMENT_HDFCBANK_DAILY.csv
  Days covered: 506 | Avg sentiment: 0.0423

Processing INFY ...
INFY: 14,691 total → 12,214 English articles (2,477 non-English removed)


...

  Article-level saved → data_1/sentiment/SENTIMENT_BHARTIARTL_ARTICLES.csv
  Daily scores saved → data_1/sentiment/SENTIMENT_BHARTIARTL_DAILY.csv
  Days covered: 599 | Avg sentiment: 0.1401

Processing HUL ...
HUL: 11,352 total → 10,958 English articles (394 non-English removed)


FinBERT batches:   0%|          | 0/343 [00:00<?, ?it/s]

  Article-level saved → data_1/sentiment/SENTIMENT_HUL_ARTICLES.csv
  Daily scores saved → data_1/sentiment/SENTIMENT_HUL_DAILY.csv
  Days covered: 1129 | Avg sentiment: 0.0727

Processing MM ...
MM: 13,024 total → 12,240 English articles (784 non-English removed)


FinBERT batches:   0%|          | 0/383 [00:00<?, ?it/s]

  Article-level saved → data_1/sentiment/SENTIMENT_MM_ARTICLES.csv
  Daily scores saved → data_1/sentiment/SENTIMENT_MM_DAILY.csv
  Days covered: 251 | Avg sentiment: 0.0990

ALL STOCKS COMPLETE


## Step 6 — Summary

In [45]:
df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))

    ticker  articles_used  days_covered  avg_sentiment
  RELIANCE          10937           409         0.0965
  HDFCBANK          10864           506         0.0423
      INFY          12214           462         0.0553
BHARTIARTL          10423           599         0.1401
       HUL          10958          1129         0.0727
        MM          12240           251         0.0990
